In [3]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [8]:
import sklearn
import pandas as pd
import logging
logging.basicConfig(level=logging.INFO)
logging.info(f"sklearn version: {sklearn.__version__}")

INFO:root:sklearn version: 1.2.2


#### **1. Loading datasets, basic feature extraction and target definitions**

**Dataset: French Motor Third-Party Liability Claims dataset. Full dataset has 678,013 samples.**

We construct the `freMTPL2` dataset by joining the `freMTPL2freq` table, containing the number of claims (`ClaimNb`), with the `freMTPL2sev` table, containing the claim amount (`ClaimAmount`) for the same policy ids (`IDpol`).

In [ ]:
def load_mtpl2(n_samples=None):
    """Fetch the French Motor Third-Party Liability Claims dataset.

    Parameters
    ----------
    n_samples: int, default=None
      number of samples to select (for faster run time). If None, all samples are selected.
    """
    # # freMTPL2freq dataset from https://www.openml.org/d/41214
    df_freq = sklearn.datasets.fetch_openml(data_id=41214, as_frame=True).data
    logging.info(f"Shape of freMTPL2freq dataset: {df_freq.shape}")
    logging.info(f"Sample of freMTPL2freq dataset:\n{df_freq.head()}")

    # Policy Ids are currently of type string, we convert them to int and set them as index for easier merging with the severity dataset.
    df_freq['IDpol'] = df_freq['IDpol'].astype(int)
    df_freq.set_index('IDpol', inplace=True)

    # freMTPL2sev dataset from https://www.openml.org/d/41215
    df_sev = sklearn.datasets.fetch_openml(data_id=41215, as_frame=True).data
    logging.info(f"Shape of freMTPL2sev dataset: {df_sev.shape}")
    logging.info(f"Sample of freMTPL2sev dataset:\n{df_sev.head()}")

    # sum claim amount over identical policy Ids (there are multiple rows per policy Id in the severity dataset, one for each claim)
    df_sev.groupby('IDpol').sum()

    # join frequency and severity datasets on policy Ids to get the final dataset for modeling
    df = df_freq.join(df_sev, how='left')

    # missing values in the claim amount column (for policies with no claims) are imputed with 0s
    df['ClaimAmount'] = df['ClaimAmount'].fillna(0)

    # unquote string fields
    for col_name in df.columns[[t is object for t in df.dtypes.values]]:
        df[col_name] = df[col_name].str.strip("'")
    
    return df.iloc[:n_samples]


In [17]:
df = load_mtpl2()
df.head()

c:\Users\Chaithra\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\datasets\_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(
INFO:root:Shape of freMTPL2freq dataset: (678013, 12)
INFO:root:Sample of freMTPL2freq dataset:
   IDpol  ClaimNb  Exposure Area  VehPower  VehAge  DrivAge  BonusMalus  \
0    1.0      1.0      0.10    D       5.0     0.0     55.0        50.0   
1    3.0      1.0      0.77    D       5.0     0.0     55.0        50.0   
2    5.0      1.0      0.75    B       6.0     2.0     52.0        50.0   
3   10.0      1.0      0.09    B       7.0     0.0     46.0        50.0   
4   11.0      1.0      0.84    B    

,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,IDpol,ClaimAmount
IDpol,,,,,,,,,,,,,
1,1.0,0.10,D,5.0,0.0,55.0,50.0,B12,Regular,1217.0,R82,1010996.0,1128.12
3,1.0,0.77,D,5.0,0.0,55.0,50.0,B12,Regular,1217.0,R82,4007252.0,1204.00
5,1.0,0.75,B,6.0,2.0,52.0,50.0,B12,Diesel,54.0,R22,4073956.0,1204.00
10,1.0,0.09,B,7.0,0.0,46.0,50.0,B12,Diesel,76.0,R72,4038917.0,77.20
11,1.0,0.84,B,7.0,0.0,46.0,50.0,B12,Diesel,76.0,R72,4078046.0,1045.00


In [ ]:
# correcting data errors
df['ClaimNb'] = df['ClaimNb'].clip(upper=4)
df['Exposure'] = df['Exposure'].clip(upper=1)
df['ClaimAmount'] = df['ClaimAmount'].clip(upper=200000)

# If the claim amount is 0, then we do not count it as a claim. The loss function
# used by the severity model needs strictly positive claim amounts. This way
# frequency and severity are more consistent with each other
df.loc[(df['ClaimAmount'] == 0) & (df['ClaimNb'] >= 1), 'ClaimNb'] = 0

df['PurePremium'] = df["ClaimAmount"] / df["Exposure"]


